In [4]:
from datasets import load_dataset
from transformers import MT5Tokenizer, MT5ForConditionalGeneration
from tqdm import tqdm
import pandas as pd
import torch
import os

# Kiểm tra GPU
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cuda" )
print(f"💻 Đang sử dụng thiết bị: {device}")

# Load model paraphrase và đưa lên GPU
CKPT = 'chieunq/vietnamese-sentence-paraphase'
tokenizer = MT5Tokenizer.from_pretrained(CKPT)
model = MT5ForConditionalGeneration.from_pretrained(CKPT).to(device)

# Hàm tạo paraphrase
def paraphrase(text, num_return_sequences=5):
    inputs = tokenizer(text, padding='longest', max_length=512, truncation=True, return_tensors='pt')
    inputs = {key: val.to(device) for key, val in inputs.items()}
    output = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=512,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_p=0.95
    )
    return [tokenizer.decode(o, skip_special_tokens=True) for o in output]

# Load dataset
dataset = pd.read_parquet("../generate_dataset/SVYKHOA_dataset_smalltalk.parquet", engine="fastparquet")

# File Excel đầu ra
output_file = "SVYKHOA_dataset_smalltalk_3.xlsx"

# Tạo file Excel rỗng nếu chưa có
if not os.path.exists(output_file):
    df_empty = pd.DataFrame(columns=["intruction", "question", "answer"])
    df_empty.to_excel(output_file, index=False)

# Đọc số dòng đã có để tiếp tục từ đó
existing_df = pd.read_excel(output_file)
# start_index = len(existing_df)
start_index = 8063
print(f"🚀 Bắt đầu từ dòng {start_index}")

# Số lượng mẫu muốn xử lý thêm
max_samples = 216350-8063           0  # Có thể chỉnh: 10, 100, 500...

# Duyệt dataset từ start_index
for i, row in dataset.iterrows():
    if i < start_index:
        continue
    if i >= start_index + max_samples:
        break
    print(row)

    question = row["question"]   # row là Series
    try:
        paraphrases = paraphrase(question, num_return_sequences=10)
    except Exception as e:
        print(f"Lỗi paraphrase tại mẫu {i}: {e}")
        paraphrases = [question]

    new_rows = []
    for pq in paraphrases:
        new_rows.append({
            "intruction": row["intruction"],
            "question": pq,
            "answer": row["answer"],
        })

    new_df = pd.DataFrame(new_rows)
    with pd.ExcelWriter(output_file, mode="a", engine="openpyxl", if_sheet_exists="overlay") as writer:
        sheet = writer.sheets["Sheet1"]
        start_row = sheet.max_row
        new_df.to_excel(writer, header=False, index=False, startrow=start_row)


print("✅ Đã lưu toàn bộ dữ liệu paraphrase vào file Excel.")


💻 Đang sử dụng thiết bị: cuda


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'T5Tokenizer'. 
The class this function is called from is 'MT5Tokenizer'.


🚀 Bắt đầu từ dòng 8063
intruction    Chatbot y khoa chuyên về y khoa hãy tư vấn về ...
question      Tôi đang cảm thấy rất căng thẳng, bạn có lời k...
answer        Căng thẳng có thể ảnh hưởng đến sức khỏe đó bạ...
Name: 8063, dtype: object
intruction    Chatbot y khoa chuyên về y khoa hãy chào tạm b...
question                                  Cảm ơn bạn đã tư vấn!
answer        Không có gì đâu bạn! 🥰 Mình rất vui vì đã giúp...
Name: 8064, dtype: object
intruction                                                 None
question      Dạ còn ạ, nếu anh chị có nhu cầu đặt hàng phiề...
answer        Dạ vâng ạ! Để em có thể gửi hàng cho anh/chị, ...
Name: 8065, dtype: object
intruction                                                 None
question            Chào bạn, tôi muốn được tư vấn về sức khỏe.
answer        😊 Xin chào! Rất vui được hỗ trợ anh/chị. Em là...
Name: 8066, dtype: object
intruction                                                 None
question                       Tại sao cầ

KeyboardInterrupt: 